In [2]:
pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/40.1 kB ? eta -:--:--
     ---------- ----------------------------- 10.2/40.1 kB ? eta -:--:--
     -------------------------------------- 40.1/40.1 kB 382.7 kB/s eta 0:00:00
     ---------------------------------------- 0.0/57.3 kB ? eta -:--:--
     --------------------- ------------------ 30.7/57.3 kB 1.3 MB/s eta 0:00:01
     -------------------------------------- 57.3/57.3 kB 747.7 kB/s eta 0:00:00
     ---------------------------------------- 0.0/42.5 kB ? eta -:--:--
     --------- ------------------------------ 10.2/42.5 kB ? eta -:--:--
     -------------------------------------- 42.5/42.5 kB 520.4 kB/s eta 0:00:00
   ---------------------------------------- 0.0/70.6 kB ? eta -:--:--
   ----------------- ---------------------- 30.7/70.6 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 70.6/70.6 kB 642.6 kB/s eta 0:00:00
   ---------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
#Download the dataset from kagglehub
import kagglehub
path = kagglehub.dataset_download("arkadiyhacks/drinking-waste-classification")
print(path)

C:\Users\SasmitaKottraivel\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1.50G/1.50G [03:37<00:00, 7.37MB/s]

Extracting files...


C:\Users\SasmitaKottraivel\.cache\kagglehub\datasets\arkadiyhacks\drinking-waste-classification\versions\2


In [6]:
import os
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')

2/
  Images_of_Waste/
    rawimgs/
      AluCan/
      Glass/
      HDPEM/
      PET/
    YOLO_imgs/


## Check the class balance — how many images are in each of the 4 rawimgs/ folders. This determines whether you need class weighting or augmentation for the smaller classes.

In [7]:
base = os.path.join(path, "Images_of_Waste", "rawimgs")
print(base)  # sanity check it exists
print(os.path.exists(base))

for cls in os.listdir(base):
    cls_path = os.path.join(base, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"{cls}: {count} images")

C:\Users\SasmitaKottraivel\.cache\kagglehub\datasets\arkadiyhacks\drinking-waste-classification\versions\2\Images_of_Waste\rawimgs
True
AluCan: 1060 images
Glass: 1224 images
HDPEM: 1028 images
PET: 1508 images


## **create a proper train/val/test split**

Right now everything sits in one folder per class. You need to physically split each class into train/, val/, test/ subfolders (stratified — same ratio from each class) so ImageFolder can load them separately without leakage.


The below code copies (not moves) files into a new dataset_split/train|val|test/<class>/ structure in your working directory — original download stays untouched.


In [8]:
import os, shutil, random
from pathlib import Path

random.seed(42)

base = os.path.join(path, "Images_of_Waste", "rawimgs")
classes = ["AluCan", "Glass", "HDPEM", "PET"]

output_base = "dataset_split"  # will be created in your current working directory
splits = {"train": 0.7, "val": 0.15, "test": 0.15}

for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(output_base, split, cls), exist_ok=True)

for cls in classes:
    cls_path = os.path.join(base, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)

    n = len(images)
    n_train = int(n * splits["train"])
    n_val = int(n * splits["val"])

    split_files = {
        "train": images[:n_train],
        "val": images[n_train:n_train + n_val],
        "test": images[n_train + n_val:]
    }

    for split, files in split_files.items():
        for f in files:
            src = os.path.join(cls_path, f)
            dst = os.path.join(output_base, split, cls, f)
            shutil.copy2(src, dst)

    print(f"{cls}: train={len(split_files['train'])}, val={len(split_files['val'])}, test={len(split_files['test'])}")

AluCan: train=742, val=159, test=159
Glass: train=856, val=183, test=185
HDPEM: train=719, val=154, test=155
PET: train=1055, val=226, test=227


In [9]:
NUM_WORKERS = 0

In [ ]:
pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports + config

In [16]:
"""
Glass / HDPE / PET / AluCan classifier
Transfer learning with ResNet18 (torchvision), fine-tuned in two phases.

Expects folder structure:
dataset_split/
  train/{AluCan,Glass,HDPEM,PET}/*.jpg
  val/{AluCan,Glass,HDPEM,PET}/*.jpg
  test/{AluCan,Glass,HDPEM,PET}/*.jpg
"""

import os
import time
import copy
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DATA_DIR = "dataset_split"          # change if your split folder is elsewhere
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4                     # set to 0 on Windows if you hit multiprocessing errors
PHASE1_EPOCHS = 5                   # train head only, backbone frozen
PHASE2_EPOCHS = 15                  # fine-tune whole network at low LR
PHASE1_LR = 1e-3
PHASE2_LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_OUT = "best_model.pt"

print(f"Using device: {DEVICE}")


Using device: cpu


## datasets + dataloaders + class weight calc (check class_names and counts print correctly)

In [17]:

# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_tfms)
val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"),   transform=eval_tfms)
test_ds  = datasets.ImageFolder(os.path.join(DATA_DIR, "test"),  transform=eval_tfms)

class_names = train_ds.classes
print("Classes (index order used by the model):", class_names)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Mild class-imbalance handling: weight loss inversely to class frequency
counts = torch.zeros(len(class_names))
for _, label in train_ds.samples:
    counts[label] += 1
class_weights = (counts.sum() / (len(class_names) * counts)).to(DEVICE)
print("Class weights:", dict(zip(class_names, class_weights.tolist())))



Classes (index order used by the model): ['AluCan', 'Glass', 'HDPEM', 'PET']
Class weights: {'AluCan': 1.1361186504364014, 'Glass': 0.9848130941390991, 'HDPEM': 1.172461748123169, 'PET': 0.7990521192550659}


## Model definition

In [18]:
# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, len(class_names))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)



Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\SasmitaKottraivel/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:09<00:00, 5.14MB/s]


## Train/eval loop functions

In [19]:
# ---------------------------------------------------------------------------
# Train / eval loops
# ---------------------------------------------------------------------------
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    torch.set_grad_enabled(is_train)
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        if is_train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def train_model(model, epochs, lr, phase_name):
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        start = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step()

        elapsed = time.time() - start
        print(f"[{phase_name}] Epoch {epoch}/{epochs} "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({elapsed:.1f}s)")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, best_val_acc



## Phase 1 training

In [20]:
# ---------------------------------------------------------------------------
# Phase 1: freeze backbone, train head only
# ---------------------------------------------------------------------------
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

model, best_val_acc = train_model(model, PHASE1_EPOCHS, PHASE1_LR, "phase1-head")



C:\Users\SasmitaKottraivel\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[phase1-head] Epoch 1/5 train_loss=1.0177 train_acc=0.6471 val_loss=0.7558 val_acc=0.8255 (171.5s)
[phase1-head] Epoch 2/5 train_loss=0.7720 train_acc=0.7998 val_loss=0.6843 val_acc=0.8518 (166.9s)
[phase1-head] Epoch 3/5 train_loss=0.7056 train_acc=0.8366 val_loss=0.6585 val_acc=0.8809 (155.9s)
[phase1-head] Epoch 4/5 train_loss=0.6747 train_acc=0.8535 val_loss=0.6433 val_acc=0.8975 (150.4s)
[phase1-head] Epoch 5/5 train_loss=0.6583 train_acc=0.8657 val_loss=0.6492 val_acc=0.8947 (162.7s)


## Phase 2 training

In [21]:
# ---------------------------------------------------------------------------
# Phase 2: unfreeze everything, fine-tune at low LR
# ---------------------------------------------------------------------------
for param in model.parameters():
    param.requires_grad = True

model, best_val_acc = train_model(model, PHASE2_EPOCHS, PHASE2_LR, "phase2-finetune")

print(f"\nBest validation accuracy: {best_val_acc:.4f}")


[phase2-finetune] Epoch 1/15 train_loss=0.5214 train_acc=0.9401 val_loss=0.4768 val_acc=0.9640 (547.7s)
[phase2-finetune] Epoch 2/15 train_loss=0.4239 train_acc=0.9870 val_loss=0.4396 val_acc=0.9861 (761.3s)
[phase2-finetune] Epoch 3/15 train_loss=0.4120 train_acc=0.9911 val_loss=0.4351 val_acc=0.9778 (653.2s)
[phase2-finetune] Epoch 4/15 train_loss=0.3942 train_acc=0.9964 val_loss=0.4312 val_acc=0.9751 (747.4s)
[phase2-finetune] Epoch 5/15 train_loss=0.3864 train_acc=0.9985 val_loss=0.4158 val_acc=0.9861 (821.0s)
[phase2-finetune] Epoch 6/15 train_loss=0.3806 train_acc=1.0000 val_loss=0.4209 val_acc=0.9834 (908.6s)
[phase2-finetune] Epoch 7/15 train_loss=0.3789 train_acc=0.9997 val_loss=0.4064 val_acc=0.9848 (864.3s)
[phase2-finetune] Epoch 8/15 train_loss=0.3737 train_acc=0.9997 val_loss=0.4075 val_acc=0.9861 (792.5s)
[phase2-finetune] Epoch 9/15 train_loss=0.3705 train_acc=1.0000 val_loss=0.4053 val_acc=0.9861 (352.7s)
[phase2-finetune] Epoch 10/15 train_loss=0.3743 train_acc=0.9994

## Save model

In [22]:

# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "img_size": IMG_SIZE,
}, MODEL_OUT)
print(f"Saved best model to {MODEL_OUT}")


Saved best model to best_model.pt


In [30]:
pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ------ --------------------------------- 10.2/61.0 kB ? eta -:--:--
     ------------------------- ------------ 41.0/61.0 kB 991.0 kB/s eta 0:00:01
     -------------------------------------- 61.0/61.0 kB 539.5 kB/s eta 0:00:00
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB 960.0 kB/s eta 0:00:09
   ---------------------------------------- 0.1/8.3 MB 871.5 kB/s eta 0:00:10
    --------------------------------------- 0.1/8.3 MB 871.5 kB/s eta 0:00:10
    --------------------------------------- 0.2/8.3 MB 833.5 kB/s eta 0:00:10
   - -------------------------------------- 0.2/8.3 MB 935.2 kB/s eta 0:00:09
   - -------------------------------------- 0.3/8.3 MB 1.0 MB/s eta 0:00:08
   - --------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Test evaluation + classification report

In [1]:

# ---------------------------------------------------------------------------
# Final test-set evaluation
# ---------------------------------------------------------------------------
test_loss, test_acc = run_epoch(model, test_loader, criterion, optimizer=None)
print(f"\nTest accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}")

# Per-class precision/recall/F1 + confusion matrix
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print("\nClassification report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("Confusion matrix (rows=true, cols=predicted):")
print(class_names)
print(confusion_matrix(all_labels, all_preds))

NameError: name 'run_epoch' is not defined

In [2]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

MODEL_PATH = "best_model.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def load_model(model_path=MODEL_PATH):
    checkpoint = torch.load(model_path, map_location=DEVICE)
    class_names = checkpoint["class_names"]
    img_size = checkpoint["img_size"]

    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, len(class_names))
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    transform = transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return model, class_names, transform

def predict(image_path, model, class_names, transform):
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.softmax(outputs, dim=1)[0]
        pred_idx = probs.argmax().item()
    return {
        "predicted_class": class_names[pred_idx],
        "confidence": round(probs[pred_idx].item(), 4),
        "all_probabilities": {class_names[i]: round(probs[i].item(), 4) for i in range(len(class_names))}
    }

In [7]:
model, class_names, transform = load_model("best_model.pt")
print("Classes:", class_names)

result = predict(r"C:\Users\SasmitaKottraivel\Downloads\GT2.jpg", model, class_names, transform)
CONFIDENCE_THRESHOLD = 0.70

if result["confidence"] < CONFIDENCE_THRESHOLD:
    print("Unknown / not confident — reject or flag for review")
else:
    print(f"Predicted: {result['predicted_class']}")

Classes: ['AluCan', 'Glass', 'HDPEM', 'PET']
Predicted: Glass


In [9]:
pip install fastapi

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/109.4 kB ? eta -:--:--
     ---------------------------------------- 0.0/109.4 kB ? eta -:--:--
     ----------- ---------------------------- 30.7/109.4 kB ? eta -:--:--
     -------------------- ---------------- 61.4/109.4 kB 812.7 kB/s eta 0:00:01
     ----------------------------- --------- 81.9/109.4 kB 1.1 MB/s eta 0:00:01
     ------------------------------------ 109.4/109.4 kB 793.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/132.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/132.0 kB ? eta -:--:--
   --------- ------------------------------ 30.7/132.0 kB 1.4 MB/s eta 0:00:01
   ------------ -------------------------- 41.0/132.0 kB 667.8 kB/s eta 0:00:01
   --------------------------- ----------- 92.2/132.0 kB 880.9 kB/s eta 0:00:01
   -------------------------------- ----- 112.6/132.0 kB 731.4 kB/s eta 0:00:01
 


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
pip install python-multipart

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
http://127.0.0.1:8000/docs

In [16]:
!python -m uvicorn app:app --host 0.0.0.0 --port 8000 --reload

^C


In [15]:
pip install uvicorn

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/79.9 kB ? eta -:--:--
   ----- ---------------------------------- 10.2/79.9 kB ? eta -:--:--
   --------------- ------------------------ 30.7/79.9 kB 259.2 kB/s eta 0:00:01
   -------------------- ------------------- 41.0/79.9 kB 245.8 kB/s eta 0:00:01
   ------------------------- -------------- 51.2/79.9 kB 217.9 kB/s eta 0:00:01
   ---------------------------------------- 79.9/79.9 kB 341.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/119.2 kB ? eta -:--:--
   --- ------------------------------------ 10.2/119.2 kB ? eta -:--:--
   ---------- ---------------------------- 30.7/119.2 kB 330.3 kB/s eta 0:00:01
   -------------------- ------------------ 61.4/119.2 kB 469.7 kB/s eta 0:00:01
   -------------------------- ------------ 81.9/119.2 kB 383.3 kB/s eta 0:00:01
   ------------------------------ -------- 92.2/119.2 kB 403.5 kB/s eta 0:00:0


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
"""
FastAPI inference service for the bottle material classifier.

Setup:
    pip install fastapi uvicorn python-multipart torch torchvision pillow

Run:
    uvicorn app:app --host 0.0.0.0 --port 8000 --reload

Test:
    curl -X POST "http://127.0.0.1:8000/predict" -F "file=@sample.jpg"

Make sure best_model.pt is in the same folder as this file (or update MODEL_PATH).
"""

import io
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
MODEL_PATH = "best_model.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Confidence below this -> treat as "unknown / not a valid bottle"
CONFIDENCE_THRESHOLD = 0.70

# ---------------------------------------------------------------------------
# Load model once at startup (not per-request — that would be slow)
# ---------------------------------------------------------------------------
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
CLASS_NAMES = checkpoint["class_names"]
IMG_SIZE = checkpoint["img_size"]

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print(f"Model loaded. Classes: {CLASS_NAMES} | Device: {DEVICE}")

# ---------------------------------------------------------------------------
# App
# ---------------------------------------------------------------------------
app = FastAPI(title="Bottle Material Classifier")

# Allow your Next.js site / ESP32-CAM to call this from any origin.
# Tighten allow_origins to your real domain once deployed.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/")
def health_check():
    return {"status": "ok", "classes": CLASS_NAMES}


@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    # Basic validation
    if not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="File must be an image")

    try:
        image_bytes = await file.read()
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    except Exception:
        raise HTTPException(status_code=400, detail="Could not read image file")

    tensor = transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.softmax(outputs, dim=1)[0]
        pred_idx = probs.argmax().item()

    predicted_class = CLASS_NAMES[pred_idx]
    confidence = round(probs[pred_idx].item(), 4)
    is_confident = confidence >= CONFIDENCE_THRESHOLD

    return {
        "class": predicted_class if is_confident else "unknown",
        "confidence": confidence,
        "accepted": is_confident,
        "all_probabilities": {
            CLASS_NAMES[i]: round(probs[i].item(), 4) for i in range(len(CLASS_NAMES))
        },
    }

Model loaded. Classes: ['AluCan', 'Glass', 'HDPEM', 'PET'] | Device: cpu
